In [ ]:
import numpy as np
import random

# Define GridWorld Environment
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.state = (0, 0)  # Start state
        self.goal = (size-1, size-1)  # Goal position

    def step(self, action):
        """Take action and return (next_state, reward)."""
        x, y = self.state
        if action == "UP":
            x = max(0, x - 1)
        elif action == "DOWN":
            x = min(self.size - 1, x + 1)
        elif action == "LEFT":
            y = max(0, y - 1)
        elif action == "RIGHT":
            y = min(self.size - 1, y + 1)

        self.state = (x, y)
        reward = 1 if self.state == self.goal else -0.1  # Reward at the goal
        return self.state, reward

    def reset(self):
        """Reset environment."""
        self.state = (0, 0)
        return self.state

# Define SARSA Agent (On-Policy RL)
class SARSAAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.env = env
        self.q_table = {}  # Q-values
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate

    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if random.random() < self.epsilon:
            return random.choice(["UP", "DOWN", "LEFT", "RIGHT"])
        return max(self.q_table.get(state, {}), key=self.q_table.get(state, {}).get, default="UP")

    def update_q_table(self, state, action, reward, next_state, next_action):
        """SARSA update."""
        if state not in self.q_table:
            self.q_table[state] = {a: 0 for a in ["UP", "DOWN", "LEFT", "RIGHT"]}
        if next_state not in self.q_table:
            self.q_table[next_state] = {a: 0 for a in ["UP", "DOWN", "LEFT", "RIGHT"]}

        self.q_table[state][action] += self.alpha * (
            reward + self.gamma * self.q_table[next_state][next_action] - self.q_table[state][action])

    def train(self, episodes=50):
        """Train the agent using SARSA."""
        for episode in range(episodes):
            state = self.env.reset()
            action = self.choose_action(state)
            while state != self.env.goal:
                next_state, reward = self.env.step(action)
                next_action = self.choose_action(next_state)
                self.update_q_table(state, action, reward, next_state, next_action)
                state, action = next_state, next_action
#            print(f"SARSA Episode {episode+1} completed!")

# Define Q-Learning Agent (Off-Policy RL)
class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.env = env
        self.q_table = {}  # Q-values
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate

    def choose_action(self, state):
        """Epsilon-greedy action selection."""
        if random.random() < self.epsilon:
            return random.choice(["UP", "DOWN", "LEFT", "RIGHT"])
        return max(self.q_table.get(state, {}), key=self.q_table.get(state, {}).get, default="UP")

    def update_q_table(self, state, action, reward, next_state):
        """Q-learning update."""
        if state not in self.q_table:
            self.q_table[state] = {a: 0 for a in ["UP", "DOWN", "LEFT", "RIGHT"]}
        if next_state not in self.q_table:
            self.q_table[next_state] = {a: 0 for a in ["UP", "DOWN", "LEFT", "RIGHT"]}

        max_next_q = max(self.q_table[next_state].values())
        self.q_table[state][action] += self.alpha * (reward + self.gamma * max_next_q - self.q_table[state][action])

    def train(self, episodes=50):
        """Train the agent using Q-learning."""
        for episode in range(episodes):
            state = self.env.reset()
            while state != self.env.goal:
                action = self.choose_action(state)
                next_state, reward = self.env.step(action)
                self.update_q_table(state, action, reward, next_state)
                state = next_state
#            print(f"Q-learning Episode {episode+1} completed!")

# Train Agents
env = GridWorld(size=5)
sarsa_agent = SARSAAgent(env)
qlearning_agent = QLearningAgent(env)

print("\nTraining SARSA (On-Policy)...")
sarsa_agent.train(episodes=50)

print("\nTraining Q-Learning (Off-Policy)...")
qlearning_agent.train(episodes=50)

# Compare Learned Q-values
print("\nLearned Q-values from SARSA (On-Policy):")
for state, actions in sarsa_agent.q_table.items():
    print(state, actions)

print("\nLearned Q-values from Q-Learning (Off-Policy):")
for state, actions in qlearning_agent.q_table.items():
    print(state, actions)
